# sql-generator: Natural Language to SQL with Claude API

This notebook demonstrates the NL-to-SQL pipeline:
1. Load the Chinook SQLite schema
2. Ask 5 natural language questions
3. Generate SQL via the Claude API (Haiku)
4. Execute and evaluate the results

**Requirements:**
- `ANTHROPIC_API_KEY` must be set in the environment
- `sql-generator/data/chinook/Chinook.sqlite` must exist

**⚠️ HUMAN APPROVAL REQUIRED** before running Cell 5 (API calls). Each question triggers one Claude API call.

In [ ]:
import pathlib
import sys
import os
from IPython.display import display
import pandas as pd

# Locate sql-generator root regardless of where the notebook is run from
_here = pathlib.Path().resolve()
if (_here / 'sql-generator').exists():      # repo root
    SQL_ROOT = _here / 'sql-generator'
elif (_here / 'src').exists():              # sql-generator/
    SQL_ROOT = _here
elif (_here.parent / 'src').exists():       # sql-generator/notebooks/
    SQL_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate sql-generator root from {_here}')

SRC = SQL_ROOT / 'src'
DB_PATH = SQL_ROOT / 'data' / 'chinook' / 'Chinook.sqlite'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'SQL_ROOT : {SQL_ROOT}')
print(f'DB exists: {DB_PATH.exists()} ({DB_PATH})')
print(f'API key  : {"set" if os.environ.get("ANTHROPIC_API_KEY") else "NOT SET — set before running Cell 5"}')

In [ ]:
from schema_loader import load_schema, format_schema_context

schema = load_schema(DB_PATH)
schema_context = format_schema_context(schema)

print(f'Tables ({len(schema)}): {sorted(schema.keys())}')
print()
print('--- Schema context (first 1500 chars) ---')
print(schema_context[:1500])

## 5 Natural Language Questions

1. List all artists alphabetically
2. How many tracks are in each genre? Order by count descending.
3. Who are the top 5 customers by total purchase amount?
4. Which albums have more than 10 tracks? Show album title and track count.
5. What is the average track length in minutes per media type?

In [ ]:
# HUMAN APPROVAL REQUIRED — this cell calls the Claude API (5 requests)
from generator import generate_sql
from executor import execute_query

questions = [
    "List all artists alphabetically",
    "How many tracks are in each genre? Order by count descending.",
    "Who are the top 5 customers by total purchase amount?",
    "Which albums have more than 10 tracks? Show album title and track count.",
    "What is the average track length in minutes per media type?",
]

results = []
for i, question in enumerate(questions, 1):
    print(f'\n--- Q{i}: {question} ---')
    try:
        sql = generate_sql(question, schema_context)
        print(f'SQL: {sql}')
        df = execute_query(sql, DB_PATH)
        display(df.head(10))
        results.append({'question': question, 'sql': sql, 'rows': len(df), 'error': None})
    except Exception as exc:
        print(f'ERROR: {exc}')
        results.append({'question': question, 'sql': None, 'rows': None, 'error': str(exc)})

In [ ]:
# Evaluate Q1: compare generated SQL against a hand-written reference
from evaluator import evaluate

reference_q1 = "SELECT Name FROM Artist ORDER BY Name"
generated_q1 = results[0]['sql'] if results else None

if generated_q1:
    eval_result = evaluate(generated_q1, reference_q1, DB_PATH)
    print('Q1 Evaluation:')
    print(f'  Exact match     : {eval_result["match"]}')
    print(f'  Results match   : {eval_result["results_match"]}')
    print(f'  Generated shape : {eval_result["generated_result_shape"]}')
    print(f'  Reference shape : {eval_result["reference_result_shape"]}')
    print(f'  Generated SQL   : {generated_q1}')
    print(f'  Reference SQL   : {reference_q1}')
else:
    print('Run Cell 5 first to generate SQL.')

In [ ]:
# Summary of all 5 results
if results:
    summary = pd.DataFrame([
        {
            'Q': f'Q{i+1}',
            'Question': r['question'][:60] + ('...' if len(r['question']) > 60 else ''),
            'SQL generated': 'Y' if r['sql'] else 'N',
            'Row count': r['rows'],
            'Error': r['error'] if r['error'] else '',
        }
        for i, r in enumerate(results)
    ])
    display(summary)
else:
    print('Run Cell 5 first to generate results.')